<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-geodevelopers/blob/main/Maps_final_version_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing the Required Libraries

In [3]:
%pip install pygeohash
!pip install folium
!pip install uszipcode
%pip install pygeohash
!pip install geopandas
!pip install folium geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 4.9 MB/s eta 0:00:00
  Created wheel for atomicwrites: filename=atomicwrites-1.4.1-py2.py3-none-any.whl size=6943 sha256=d2f71751111e7a6859d2c26d2a97abaf343b6d07a850dba2d126c508adc7a99d
  Stored in directory: /root/.cache/pip/wheels/f7/99/9c/d24e98c35f30eba0c367ad1e7888d396d676abb35fe1e7611c
Successfully built atomicwrites


In [4]:
import pandas as pd
import geopandas as gpd
import folium
import pygeohash as gh
import numpy as np

In [5]:
from datascience import *
%matplotlib inline
#path_data = '../../../assets/data/'
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
from matplotlib import colors

In [6]:
#Select the geohash percision
Geohash_percision = 6
#Select the required column to be used for clustering and MST pollution weight
Analyzing_Pollution_Column = "TotalNormalizedPollution"
#Create new dataframe table, and import all available Chicago files in the repositry to it.
Chicago_df_original = Table().to_df()

for i in range(1,20):
  file_link = 'https://raw.githubusercontent.com/IsamAljawarneh/datasets/refs/heads/master/data/Chicago/AQ_data/chicago_eclipse_data_part_'+str(i)+'.csv'
  Chicago_df_original = pd.concat([Chicago_df_original, pd.read_csv(file_link,index_col=False)], ignore_index=True)
#.show() is a method of datascience.Table, not pandas.DataFrame that why you are using head instad of show
Chicago_df_original.head(5)

,City,DeviceId,LocationName,Latitude,Longitude,ReadingDateTimeUTC,PM25,CalibratedPM25,CalibratedO3,CalibratedNO2,CO,Temperature,Humidity,BatteryLevel,PercentBattery,CellSignal
0,Chicago,2002,State & Garfield (SB),41.794921,-87.625857,2021-06-20 00:03:00,5.561094,NaN,NaN,NaN,0.123580,27.383499,55.128479,4.237187,93.964844,-76.0
1,Chicago,2002,State & Garfield (SB),41.794921,-87.625857,2021-06-20 00:08:10,6.633914,NaN,NaN,NaN,0.132103,27.079086,55.059814,4.236094,93.964844,-81.0
2,Chicago,2002,State & Garfield (SB),41.794921,-87.625857,2021-06-20 00:13:20,4.068707,NaN,NaN,NaN,0.131126,27.079086,55.035400,4.236406,93.964844,-80.0
3,Chicago,2002,State & Garfield (SB),41.794921,-87.625857,2021-06-20 00:18:30,6.351702,NaN,NaN,NaN,0.138784,26.945572,54.632568,4.236094,93.863281,-82.0
4,Chicago,2002,State & Garfield (SB),41.794921,-87.625857,2021-06-20 00:23:40,9.574065,NaN,NaN,NaN,0.413070,26.828079,53.907776,4.235938,93.863281,-81.0


Generate geohash for each (Latitude, Longitude) pair

In [ ]:
%%time
Chicago_df_original['geohash'] = Chicago_df_original.apply(
    lambda x: gh.encode(x['Latitude'], x['Longitude'], precision=Geohash_percision),axis=1)
#Chicago_df_original.select('City','Latitude','Longitude','geohash')
Chicago_df_original[['City', 'Latitude', 'Longitude', 'geohash']]


converting geopandas to geodataframe

In [ ]:
%%time
gdf_chicago = gpd.GeoDataFrame(
    Chicago_df_original,geometry=gpd.points_from_xy(Chicago_df_original.Longitude, Chicago_df_original.Latitude)
)
#GeoDataFrame
#gdf_chicago.head()

In [2]:
# Set the CRS to WGS 84 (EPSG:4326)
gdf_chicago = gdf_chicago.set_crs('epsg:4326')

# Show first few rows
gdf_chicago.head()

NameError: name 'gdf_chicago' is not defined

***Note: This GeoJSON file is from an external source (not our own) ***

In [ ]:
# Chicago neighborhood polygons GeoJSON
geojson_file = "https://raw.githubusercontent.com/blackmad/neighborhoods/refs/heads/master/chicago.geojson"

# Read the GeoJSON file into a GeoDataFrame
neighborhoods_chicago = gpd.read_file(geojson_file)

In [ ]:
neighborhoods_chicago.crs

spatial join

In [ ]:

sjoined_chicago = gpd.sjoin(gdf_chicago, neighborhoods_chicago, predicate="within")

sjoined_chicago

In [ ]:
# Step 5: Count unique neighborhoods
n = len(pd.unique(sjoined_chicago['LocationName']))
print("Number of unique neighborhood values:", n)

# Optional: Check type and shape
print(type(sjoined_chicago))
print(sjoined_chicago.shape)


In [ ]:
# Stratified sampling by 'neighborhood' by group
sampled_geohash_data_base = sjoined_chicago.groupby('LocationName').apply(lambda x: x.sample(frac=0.6))

In [ ]:
sampled_geohash_data_base.shape

In [ ]:
sampled_geohash_data_base

**Choropleth Maps**

In [ ]:
#Count occurrences of neighborhoods in the sampled data
chicago_pickup_sample2 = sampled_geohash_data_base['LocationName'].value_counts()
chicago_pickup_sample2 = chicago_pickup_sample2.reset_index()
chicago_pickup_sample2.columns = ['LocationName', 'count']
chicago_pickup_sample2['LocationName'] = chicago_pickup_sample2['LocationName'].astype(str)
# Step 2: Count occurrences in the original joined data
chicago_pickup_original = sjoined_chicago['LocationName'].value_counts()
chicago_pickup_original = chicago_pickup_original.reset_index()
chicago_pickup_original.columns = ['LocationName', 'count']
chicago_pickup_original['LocationName'] = chicago_pickup_original['LocationName'].astype(str)


In [ ]:
chicago_pickup_original

In [ ]:
chicago_pickup_sample2

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import requests

# Step 1: Use the existing spatially joined GeoDataFrame
# This assumes you've already done:
# sjoined_chicago = gpd.sjoin(gdf_chicago, neighborhoods_chicago, predicate="within")

sensors_df = sjoined_chicago.copy()

# Step 2: Load Chicago neighborhoods GeoJSON
geojson_url = 'https://raw.githubusercontent.com/blackmad/neighborhoods/refs/heads/master/chicago.geojson'
chicago_gdf = gpd.read_file(geojson_url)

# Step 3: Aggregate PM2.5 values by neighborhood
chicago_pollution_df = sensors_df.groupby('name')['PM25'].mean().reset_index()

# Step 4: Load GeoJSON as a dictionary for folium
geo_data = requests.get(geojson_url).json()

# Step 5: Merge PM2.5 data into GeoJSON
for feature in geo_data['features']:
    hood_name = feature['properties']['name']
    match = chicago_pollution_df[chicago_pollution_df['name'] == hood_name]
    feature['properties']['PM25'] = float(match['PM25'].values[0]) if not match.empty else None

# Step 6: Create base map
map_chicago = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Step 7: Add choropleth for legend
folium.Choropleth(
    geo_data=geojson_url,
    data=chicago_pollution_df,
    columns=['name', 'PM25'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.5,
    line_opacity=0.2,
    bins=[0, 5, 10, 15, 25, 35, 50],
    nan_fill_color='gray',
    legend_name='Average PM2.5 (µg/m³)'
).add_to(map_chicago)

# Step 8: Custom style function + tooltips
def style_function(feature):
    value = feature['properties'].get('PM25')
    if value is None:
        return {'fillColor': 'gray', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
    elif value < 5:
        return {'fillColor': '#ffffb2', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 10:
        return {'fillColor': '#fecc5c', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 15:
        return {'fillColor': '#fd8d3c', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 25:
        return {'fillColor': '#f03b20', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    else:
        return {'fillColor': '#bd0026', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}

folium.GeoJson(
    geo_data,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'PM25'], aliases=['Neighborhood:', 'PM2.5:'])
).add_to(map_chicago)

# Step 9: Save and show
map_chicago.save("chicago_pm25_map.html")
map_chicago


map for CalibratedPM25

In [ ]:

# Group by neighborhood and calculate the mean CalibratedPM25
chicago_calibrated_df = sensors_df.groupby('name')['CalibratedPM25'].mean().reset_index()

# Merge CalibratedPM25 values into GeoJSON
for feature in geo_data['features']:
    hood_name = feature['properties']['name']
    match = chicago_calibrated_df[chicago_calibrated_df['name'] == hood_name]
    feature['properties']['CalibratedPM25'] = float(match['CalibratedPM25'].values[0]) if not match.empty else None

# Create a new map for calibrated data
map_chicago_calibrated = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Add choropleth for CalibratedPM25 (for legend only)
folium.Choropleth(
    geo_data=geojson_url,
    data=chicago_calibrated_df,
    columns=['name', 'CalibratedPM25'],
    key_on='feature.properties.name',
    fill_color='YlGnBu',
    fill_opacity=0.5,
    line_opacity=0.2,
    bins=[0, 5, 10, 15, 25, 35, 50],
    nan_fill_color='gray',
    legend_name='Average Calibrated PM2.5 (µg/m³)'
).add_to(map_chicago_calibrated)

# Style function for CalibratedPM25
def style_calibrated(feature):
    value = feature['properties'].get('CalibratedPM25')
    if value is None:
        return {'fillColor': 'gray', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
    elif value < 5:
        return {'fillColor': '#edf8b1', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 10:
        return {'fillColor': '#7fcdbb', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 15:
        return {'fillColor': '#41b6c4', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 25:
        return {'fillColor': '#1d91c0', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    else:
        return {'fillColor': '#225ea8', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}

# Add GeoJSON with calibrated PM2.5 tooltips
folium.GeoJson(
    geo_data,
    style_function=style_calibrated,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'CalibratedPM25'], aliases=['Neighborhood:', 'Calibrated PM2.5:'])
).add_to(map_chicago_calibrated)

# Save the calibrated map
map_chicago_calibrated.save("chicago_calibrated_pm25_map.html")
map_chicago_calibrated  # Display in notebook


In [ ]:
# Step 11: Create map for CalibratedO3
# Group by neighborhood and calculate the mean CalibratedO3
chicago_o3_df = sensors_df.groupby('name')['CalibratedO3'].mean().reset_index()

# Merge CalibratedO3 values into GeoJSON
for feature in geo_data['features']:
    hood_name = feature['properties']['name']
    match = chicago_o3_df[chicago_o3_df['name'] == hood_name]
    feature['properties']['CalibratedO3'] = float(match['CalibratedO3'].values[0]) if not match.empty else None

# Create a new map for CalibratedO3
map_chicago_o3 = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Add choropleth for CalibratedO3
folium.Choropleth(
    geo_data=geojson_url,
    data=chicago_o3_df,
    columns=['name', 'CalibratedO3'],
    key_on='feature.properties.name',
    fill_color='PuBuGn',
    fill_opacity=0.5,
    line_opacity=0.2,
    bins=[0, 10, 20, 30, 40, 50, 60],
    nan_fill_color='gray',
    legend_name='Average Calibrated O₃ (ppb)'
).add_to(map_chicago_o3)

# Style function for CalibratedO3
def style_o3(feature):
    value = feature['properties'].get('CalibratedO3')
    if value is None:
        return {'fillColor': 'gray', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
    elif value < 10:
        return {'fillColor': '#f7fcfd', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 20:
        return {'fillColor': '#e0ecf4', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 30:
        return {'fillColor': '#bfd3e6', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 40:
        return {'fillColor': '#9ebcda', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 50:
        return {'fillColor': '#8c96c6', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    else:
        return {'fillColor': '#8856a7', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}

# Add GeoJson layer with tooltip for CalibratedO3
folium.GeoJson(
    geo_data,
    style_function=style_o3,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'CalibratedO3'], aliases=['Neighborhood:', 'Calibrated O₃:'])
).add_to(map_chicago_o3)

# Save and show map
map_chicago_o3.save("chicago_calibrated_o3_map.html")
map_chicago_o3


In [ ]:
# Step 12: Create map for CalibratedNO2
# Group by neighborhood and calculate the mean CalibratedNO2
chicago_no2_df = sensors_df.groupby('name')['CalibratedNO2'].mean().reset_index()

# Merge CalibratedNO2 values into GeoJSON
for feature in geo_data['features']:
    hood_name = feature['properties']['name']
    match = chicago_no2_df[chicago_no2_df['name'] == hood_name]
    feature['properties']['CalibratedNO2'] = float(match['CalibratedNO2'].values[0]) if not match.empty else None

# Create a new map for CalibratedNO2
map_chicago_no2 = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Add choropleth for CalibratedNO2
folium.Choropleth(
    geo_data=geojson_url,
    data=chicago_no2_df,
    columns=['name', 'CalibratedNO2'],
    key_on='feature.properties.name',
    fill_color='YlGnBu',
    fill_opacity=0.5,
    line_opacity=0.2,
    bins=[0, 5, 10, 15, 20, 30, 50],
    nan_fill_color='gray',
    legend_name='Average Calibrated NO₂ (ppb)'
).add_to(map_chicago_no2)

# Style function for CalibratedNO2
def style_no2(feature):
    value = feature['properties'].get('CalibratedNO2')
    if value is None:
        return {'fillColor': 'gray', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
    elif value < 5:
        return {'fillColor': '#ffffcc', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 10:
        return {'fillColor': '#a1dab4', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 15:
        return {'fillColor': '#41b6c4', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 20:
        return {'fillColor': '#2c7fb8', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    else:
        return {'fillColor': '#253494', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}

# Add styled GeoJson layer with tooltip for CalibratedNO2
folium.GeoJson(
    geo_data,
    style_function=style_no2,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'CalibratedNO2'], aliases=['Neighborhood:', 'Calibrated NO₂:'])
).add_to(map_chicago_no2)

#To Save and show map
map_chicago_no2.save("chicago_calibrated_no2_map.html")
map_chicago_no2


In [ ]:
# Step: Group by neighborhood and calculate the mean Temperature
chicago_temp_df = sensors_df.groupby('name')['Temperature'].mean().reset_index()

# Merge Temperature values into GeoJSON
for feature in geo_data['features']:
    hood_name = feature['properties']['name']
    match = chicago_temp_df[chicago_temp_df['name'] == hood_name]
    feature['properties']['Temperature'] = float(match['Temperature'].values[0]) if not match.empty else None

# Create a new map for Temperature
map_chicago_temp = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Add choropleth for Temperature
folium.Choropleth(
    geo_data=geojson_url,
    data=chicago_temp_df,
    columns=['name', 'Temperature'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0.3,
    bins=[0, 10, 15, 20, 25, 30, 35, 40],  # adjust as needed
    nan_fill_color='gray',
    legend_name='Average Temperature (°C)'  # adjust unit if needed
).add_to(map_chicago_temp)

# Style function for Temperature
def style_temp(feature):
    value = feature['properties'].get('Temperature')
    if value is None:
        return {'fillColor': 'gray', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.5}
    elif value < 10:
        return {'fillColor': '#ffffb2', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 15:
        return {'fillColor': '#fecc5c', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 20:
        return {'fillColor': '#fd8d3c', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    elif value < 25:
        return {'fillColor': '#f03b20', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}
    else:
        return {'fillColor': '#bd0026', 'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7}

# Add styled GeoJson layer with tooltip for Temperature
folium.GeoJson(
    geo_data,
    style_function=style_temp,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'Temperature'], aliases=['Neighborhood:', 'Temperature (°C):'])
).add_to(map_chicago_temp)

# Save and show the map
map_chicago_temp.save("chicago_temperature_map.html")
map_chicago_temp


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import geopandas as gpd

# Step 1: Use spatially joined data (assuming 'sjoined_chicago' exists)
sensors_df = sjoined_chicago.copy()

# Step 2: Filter rows with valid Temperature and coordinate data
heat_data = sensors_df[['Latitude', 'Longitude', 'Temperature']].dropna()

# Step 3: Create list of [lat, lon, weight] for heatmap
heat_points = heat_data[['Latitude', 'Longitude', 'Temperature']].values.tolist()

# Step 4: Create base map
heatmap_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Step 5: Add heatmap layer
HeatMap(
    heat_points,
    radius=20,         # Size of heat "blobs"
    blur=15,           # Smoothing
    min_opacity=0.4,   # Adjust fogginess
    max_zoom=1
).add_to(heatmap_map)

# Step 6: Save and display
heatmap_map.save("chicago_temperature_true_heatmap.html")
heatmap_map
